# FINAL 06 v2 — RQ4 External Robustness (frozen 100k Sentiment140 sample)

**Thesis:** An Ethical Risk Assessment of Sentiment Analysis Models in Social Media Monitoring with a Focus on Bias, Inclusivity, and Misclassification Patterns  
**Student:** Abhishek Tomar | PN1196973 | LJMU | October 2026

**Change from FINAL_06 v1 (only change):** the Sentiment140 target is now the frozen EDA_03 v3 evaluation sample
(`df_sentiment140_eval_sample_final.parquet`: 100,000 unique-text rows, 50,000 per class, conflicting-label duplicates removed)
instead of `LEGACY/data/s140_test.parquet` (10,000 rows). Subgroups are tagged on raw `text`; the model is evaluated on `text_eval`
(label-source-like ASCII emoticons removed), exactly as frozen in `sentiment140_role_freeze.json`.

**Unchanged:** locked FINAL_02 pipeline, SemEval-2014 input and logic, binary evaluation logic (neutral predictions = errors),
matched TweetEval binary comparator, frozen subgroup rules, N ≥ 30 rule, output file names consumed by FINAL_07.

> **Final-run rule:** TweetEval test data is held out from model/feature/hyperparameter selection. External datasets are never used for source-model selection. Linguistic subgroup thresholds are frozen and are not tuned after observing subgroup sizes.

In [1]:
!pip install -q emoji vaderSentiment pandas pyarrow scikit-learn scipy google-api-python-client

from google.colab import drive, auth
drive.mount('/content/drive')

import io, sys, json, hashlib, shutil, importlib
from pathlib import Path
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

UTIL_FILE_ID = "14KfwtUf2zSd_jlW7i-6K8XgBJ62Aqk02"            # unchanged from v1
S140_FROZEN_FILE_ID = "1Q3FkyjY3tMcAJT6l0Fn9F7MBU5z6MjSk"     # EDA/Results/Sentiment140/df_sentiment140_eval_sample_final.parquet
S140_FROZEN_BYTES = 17246942

auth.authenticate_user()
creds, _ = google.auth.default()
service = build("drive", "v3", credentials=creds, cache_discovery=False)

def drive_download(file_id, local_path):
    local_path = Path(local_path)
    with io.FileIO(str(local_path), "wb") as fh:
        dl = MediaIoBaseDownload(fh, service.files().get_media(fileId=file_id))
        done = False
        while not done:
            _, done = dl.next_chunk()
    assert local_path.exists() and local_path.stat().st_size > 0, f"download failed: {file_id}"
    return local_path

UTIL_LOCAL = drive_download(UTIL_FILE_ID, "/content/final_thesis_utils.py")
if "/content" not in sys.path: sys.path.insert(0, "/content")
import final_thesis_utils as ftu
ftu = importlib.reload(ftu)
RUN_ROOT = Path("/content/drive/MyDrive/LJMU_FINAL_EXPERIMENT_RUN")
ftu.FINAL_ROOT = RUN_ROOT
ftu.FINAL = {k: RUN_ROOT / v for k, v in
             {"data":"data","models":"models","predictions":"predictions","results":"results","figures":"figures","manifests":"manifests"}.items()}
ftu.FINAL["root"] = RUN_ROOT
globals().update({k: v for k, v in vars(ftu).items() if not k.startswith("_")})
FINAL_ROOT, FINAL = ftu.FINAL_ROOT, ftu.FINAL
setup_final_dirs()

import numpy as np, pandas as pd, sklearn
print("sklearn", sklearn.__version__)

MODEL_PATH = FINAL["models"] / "FINAL_SELECTED_PIPELINE.pkl"
assert MODEL_PATH.exists(), f"Missing FINAL_02 locked model: {MODEL_PATH}"
final_pipe = load_pickle(MODEL_PATH)
MODEL_SHA256 = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
print("Locked model sha256:", MODEL_SHA256)

SOURCE_PRED_PATH = FINAL["predictions"] / "FINAL_SELECTED_TweetEval.parquet"
source_pred = pd.read_parquet(SOURCE_PRED_PATH)
assert len(source_pred) == 12284
print("Loaded locked pipeline and 12,284 TweetEval source predictions.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.6 MB/s eta 0:00:00
Mounted at /content/drive


sklearn 1.6.1
Locked model sha256: bf92c6162dd4664c4a3445d8ce806cc28b09809f75613d6c70f4ef8b6ad954e2
Loaded locked pipeline and 12,284 TweetEval source predictions.


## 1. Archive the v1 (legacy 10k) outputs before they are overwritten
The v1 files are kept, renamed with a `_v1_legacy10k` suffix, so the change is fully traceable.

In [2]:
V1_FILES = [
    FINAL["results"] / "F09_Sentiment140_binary_robustness.csv",
    FINAL["results"] / "F09_F10_external_subgroup_coverage.csv",
    FINAL["results"] / "F09_F10_external_subgroup_performance.csv",
    FINAL["results"] / "F10_SemEval_domain_task_shift.csv",
    FINAL["predictions"] / "F09_FinalSelected_Sentiment140.parquet",
    FINAL["predictions"] / "F10_FinalSelected_SemEval.parquet",
    FINAL["results"] / "F12_ethical_risk_evidence_profile.csv",
]
for p in V1_FILES:
    if p.exists():
        dst = p.with_name(p.stem + "_v1_legacy10k" + p.suffix)
        if not dst.exists():
            shutil.copy2(p, dst)
        print("archived:", dst.name)
    else:
        print("not found (ok if first run):", p.name)

archived: F09_Sentiment140_binary_robustness_v1_legacy10k.csv
archived: F09_F10_external_subgroup_coverage_v1_legacy10k.csv
archived: F09_F10_external_subgroup_performance_v1_legacy10k.csv
archived: F10_SemEval_domain_task_shift_v1_legacy10k.csv
archived: F09_FinalSelected_Sentiment140_v1_legacy10k.parquet
archived: F10_FinalSelected_SemEval_v1_legacy10k.parquet
archived: F12_ethical_risk_evidence_profile_v1_legacy10k.csv


## 2. Load external targets
SemEval-2014 is unchanged (legacy `se_test.parquet`, 1,758 aspect-level rows). Sentiment140 is the frozen EDA sample.

In [3]:
se = pd.read_parquet(LEGACY["data"] / "se_test.parquet").copy()

S140_LOCAL = drive_download(S140_FROZEN_FILE_ID, "/content/df_sentiment140_eval_sample_final.parquet")
assert S140_LOCAL.stat().st_size == S140_FROZEN_BYTES, f"Unexpected file size: {S140_LOCAL.stat().st_size}"
S140_SHA256 = hashlib.sha256(S140_LOCAL.read_bytes()).hexdigest()
s140_raw = pd.read_parquet(S140_LOCAL)
print("Sentiment140 frozen sample columns:", list(s140_raw.columns))

# Integrity checks against the EDA_03 v3 freeze
assert len(s140_raw) == 100_000, len(s140_raw)
assert s140_raw["sentiment"].value_counts().to_dict() == {"positive": 50_000, "negative": 50_000} or \
       sorted(s140_raw["sentiment"].value_counts().tolist()) == [50_000, 50_000]
assert {"text", "text_eval", "sentiment"}.issubset(s140_raw.columns)
assert s140_raw["text"].str.strip().str.lower().str.replace(r"\s+", " ", regex=True).is_unique, "frozen sample should be unique-text"
print("Frozen Sentiment140 sample verified: 100,000 rows, 50k/50k, unique normalised text. sha256:", S140_SHA256)

def normalize_external(df):
    x = df.copy()
    if "sentiment" not in x.columns:
        for c in ["label_text", "sentiment_label"]:
            if c in x.columns: x["sentiment"] = x[c]
    x["sentiment"] = x["sentiment"].astype(str).str.lower()
    # drop any EDA-era tagging columns so the FINAL utility re-tags from raw text
    x = x.drop(columns=[c for c in x.columns if c in
        {"subgroup","word_count","emoji_count","slang_count","emoji_density","slang_ratio",
         "is_emoji_heavy","is_slang_heavy","is_sarcasm_indicated","is_reference_candidate"}], errors="ignore")
    x = tag_subgroups(x, "text")              # subgroup tagging on RAW text (frozen rule)
    x["text_clean"] = x["text"].map(clean_text)
    return x

se = normalize_external(se)
s140 = normalize_external(s140_raw)

# The FINAL utility must reproduce the frozen EDA_03 coverage exactly
EXPECTED_S140 = {"reference-unmarked": 73512, "other": 23450, "slang-heavy": 2795,
                 "sarcasm-indicated": 171, "emoji-heavy": 72}
got = s140["subgroup"].value_counts().to_dict()
assert got == EXPECTED_S140, f"Subgroup counts differ from EDA freeze: {got}"
print("SemEval", len(se), se.sentiment.value_counts().to_dict())
print("Sentiment140", len(s140), s140.sentiment.value_counts().to_dict())
print("Sentiment140 subgroups (match EDA freeze):", got)

Sentiment140 frozen sample columns: ['text', 'label', 'sentiment', 'text_norm', 'has_pos_source_like_emoticon', 'has_neg_source_like_emoticon', 'has_source_like_emoticon', 'text_eval', 'source_like_emoticon_removed', 'text_whitespace_normalized', 'word_count', 'emoji_count', 'slang_count', 'emoji_density', 'slang_ratio', 'is_emoji_heavy', 'is_slang_heavy', 'is_sarcasm_indicated', 'is_reference_candidate', 'subgroup']
Frozen Sentiment140 sample verified: 100,000 rows, 50k/50k, unique normalised text. sha256: d25327ccb390945e945f687c38f84e3973e81199c7b7977f02d1f5a08593b8ed
SemEval 1758 {'positive': 1069, 'neutral': 365, 'negative': 324}
Sentiment140 100000 {'positive': 50000, 'negative': 50000}
Sentiment140 subgroups (match EDA freeze): {'reference-unmarked': 73512, 'other': 23450, 'slang-heavy': 2795, 'sarcasm-indicated': 171, 'emoji-heavy': 72}


## 3. SemEval-2014 domain/task-shift condition (unchanged from v1)
Evaluation unit: aspect-level rows. One sentence can carry several annotated aspects, so 1,758 aspect rows correspond to fewer unique sentences.

In [4]:
from sklearn.metrics import f1_score, accuracy_score

se3 = se[se.sentiment.isin(LABELS)].copy()
yse = se3.sentiment.map(LABEL_TO_ID).astype(int).to_numpy()
ypse = final_pipe.predict(se3.text.astype(str))
ppse = final_pipe.predict_proba(se3.text.astype(str))
se3["_y_true"] = yse; se3["_y_pred"] = ypse

se_metrics = evaluate_predictions(yse, ypse, ppse, "FinalSelectedPipeline")
source_3class_acc = accuracy_score(source_pred["y_true"], source_pred["y_pred"])
source_3class_f1 = f1_score(source_pred["y_true"], source_pred["y_pred"], labels=[0,1,2], average="macro", zero_division=0)
se_metrics.update({
    "Target Dataset": "SemEval-2014", "Condition": "domain/task-shift stress",
    "Evaluation Unit": "aspect-level row",
    "Target N": len(se3), "Unique Sentences": int(se3.text.str.strip().str.lower().nunique()),
    "TweetEval Source 3-Class N": len(source_pred),
    "TweetEval Source 3-Class Accuracy": source_3class_acc,
    "TweetEval Source 3-Class Macro F1": source_3class_f1,
    "Accuracy Delta vs TweetEval Source": se_metrics["Accuracy"] - source_3class_acc,
    "Macro F1 Delta vs TweetEval Source": se_metrics["Macro F1"] - source_3class_f1,
})
pd.DataFrame([se_metrics]).to_csv(FINAL["results"] / "F10_SemEval_domain_task_shift.csv", index=False)
prediction_frame(se3, yse, ypse, ppse, "FinalSelectedPipeline", "SemEval-2014").to_parquet(
    FINAL["predictions"] / "F10_FinalSelected_SemEval.parquet", index=False)

# v1 regression guard: SemEval must be identical to v1 because nothing changed here
assert abs(se_metrics["Macro F1"] - 0.5735511881960479) < 1e-9, se_metrics["Macro F1"]
print(se_metrics)

{'Model': 'FinalSelectedPipeline', 'Accuracy': 0.6484641638225256, 'Macro F1': 0.5735511881960479, 'Weighted F1': 0.6622794276729448, 'Macro Precision': 0.6104349483915746, 'Macro Recall': 0.576668704319904, 'MCC': np.float64(0.4030306357987246), 'Cohen Kappa': np.float64(0.3929747947238327), 'Brier': 0.14990148867478664, 'Target Dataset': 'SemEval-2014', 'Condition': 'domain/task-shift stress', 'Evaluation Unit': 'aspect-level row', 'Target N': 1758, 'Unique Sentences': 1011, 'TweetEval Source 3-Class N': 12284, 'TweetEval Source 3-Class Accuracy': 0.6213774014978835, 'TweetEval Source 3-Class Macro F1': 0.6053146348313904, 'Accuracy Delta vs TweetEval Source': 0.027086762324642133, 'Macro F1 Delta vs TweetEval Source': -0.031763446635342474}


## 4. Sentiment140 binary robustness (frozen 100k sample)
Truth labels negative/positive; the locked three-class pipeline may still predict neutral, which counts as an error on both source and target. Prediction input is `text_eval`.

In [5]:
sbin = s140[s140.sentiment.isin(["negative", "positive"])].copy()
true_bin = sbin.sentiment.map({"negative": 0, "positive": 2}).astype(int).to_numpy()
EVAL_TEXT = sbin["text_eval"].astype(str)            # emoticon-stripped evaluation text (frozen policy)
yp = final_pipe.predict(EVAL_TEXT)
pp = final_pipe.predict_proba(EVAL_TEXT)
sbin["_y_true"] = true_bin; sbin["_y_pred"] = yp

src_bin = source_pred[source_pred["y_true"].isin([0, 2])].copy()
src_bin_acc = accuracy_score(src_bin["y_true"], src_bin["y_pred"])
src_bin_f1 = f1_score(src_bin["y_true"], src_bin["y_pred"], labels=[0, 2], average="macro", zero_division=0)
src_bin_neutral_rate = float((src_bin["y_pred"] == 1).mean())

target_acc = accuracy_score(true_bin, yp)
target_f1 = f1_score(true_bin, yp, labels=[0, 2], average="macro", zero_division=0)
target_neutral_rate = float((yp == 1).mean())

s140_metrics = {
    "Model": "FinalSelectedPipeline", "Target Dataset": "Sentiment140",
    "Condition": "binary external robustness",
    "Target Sample": "EDA_03 v3 frozen 100k unique-text sample (50k/50k)",
    "Target N": len(sbin), "Accuracy": target_acc,
    "Macro F1 (negative/positive truth labels)": target_f1,
    "Neutral Prediction Rate": target_neutral_rate,
    "TweetEval Source Binary N": len(src_bin),
    "TweetEval Source Binary Accuracy": src_bin_acc,
    "TweetEval Source Binary Macro F1": src_bin_f1,
    "TweetEval Source Binary Neutral Prediction Rate": src_bin_neutral_rate,
    "Accuracy Delta vs TweetEval Source Binary": target_acc - src_bin_acc,
    "Macro F1 Delta vs TweetEval Source Binary": target_f1 - src_bin_f1,
    "Neutral Prediction Rate Delta vs TweetEval Source Binary": target_neutral_rate - src_bin_neutral_rate,
}
# the matched TweetEval comparator must be identical to v1
assert len(src_bin) == 6347 and abs(src_bin_f1 - 0.677815530444217) < 1e-9
pd.DataFrame([s140_metrics]).to_csv(FINAL["results"] / "F09_Sentiment140_binary_robustness.csv", index=False)
prediction_frame(sbin.assign(text=EVAL_TEXT), true_bin, yp, pp, "FinalSelectedPipeline", "Sentiment140").to_parquet(
    FINAL["predictions"] / "F09_FinalSelected_Sentiment140.parquet", index=False)
print(json.dumps(s140_metrics, indent=2, default=float))

{
  "Model": "FinalSelectedPipeline",
  "Target Dataset": "Sentiment140",
  "Condition": "binary external robustness",
  "Target Sample": "EDA_03 v3 frozen 100k unique-text sample (50k/50k)",
  "Target N": 100000,
  "Accuracy": 0.49134,
  "Macro F1 (negative/positive truth labels)": 0.5803732949377841,
  "Neutral Prediction Rate": 0.33526,
  "TweetEval Source Binary N": 6347,
  "TweetEval Source Binary Accuracy": 0.5388372459429652,
  "TweetEval Source Binary Macro F1": 0.677815530444217,
  "TweetEval Source Binary Neutral Prediction Rate": 0.39152355443516623,
  "Accuracy Delta vs TweetEval Source Binary": -0.04749724594296523,
  "Macro F1 Delta vs TweetEval Source Binary": -0.09744223550643294,
  "Neutral Prediction Rate Delta vs TweetEval Source Binary": -0.05626355443516623
}


## 5. Coverage-qualified external subgroup analysis (logic unchanged)

In [6]:
coverage, subgroup_rows = [], []
for dsname, df, labels_eval, condition in [
    ("SemEval-2014", se3, [0, 1, 2], "domain/task-shift stress"),
    ("Sentiment140", sbin, [0, 2], "binary external robustness"),
]:
    for sg in FOCAL_GROUPS:
        part = df[df.subgroup == sg].copy(); n = len(part); eligible = n >= MIN_STABLE_N
        coverage.append({"Dataset": dsname, "Subgroup": sg, "N": n,
                         "Eligible for stable subgroup interpretation": eligible})
        if n:
            row = {"Dataset": dsname, "Condition": condition, "Subgroup": sg, "N": n,
                   "Accuracy": accuracy_score(part["_y_true"], part["_y_pred"]),
                   "Macro F1": f1_score(part["_y_true"], part["_y_pred"], labels=labels_eval, average="macro", zero_division=0),
                   "Interpretation Status": "stable_quantitative" if eligible else "exploratory_descriptive_only"}
            if dsname == "Sentiment140":
                row["Neutral Prediction Rate"] = float((part["_y_pred"] == 1).mean())
            else:
                row["Neutral Prediction Rate"] = np.nan
            subgroup_rows.append(row)

coverage = pd.DataFrame(coverage)
coverage.to_csv(FINAL["results"] / "F09_F10_external_subgroup_coverage.csv", index=False)
external_subgroups = pd.DataFrame(subgroup_rows)
external_subgroups["Macro F1 Gap vs Reference"] = np.nan
for dsname in external_subgroups["Dataset"].unique():
    ref = external_subgroups[(external_subgroups["Dataset"] == dsname) & (external_subgroups["Subgroup"] == REFERENCE_GROUP)]
    if len(ref):
        mask = external_subgroups["Dataset"] == dsname
        external_subgroups.loc[mask, "Macro F1 Gap vs Reference"] = external_subgroups.loc[mask, "Macro F1"] - float(ref.iloc[0]["Macro F1"])

# Binary-label bootstrap 95% CI for Sentiment140 subgroup Macro F1 (500 resamples, seed 42)
def boot_ci_binary(y, p, n_boot=500, seed=SEED):
    rng = np.random.default_rng(seed); n = len(y); vals = []
    for _ in range(n_boot):
        i = rng.integers(0, n, n); vals.append(f1_score(y[i], p[i], labels=[0, 2], average="macro", zero_division=0))
    return tuple(np.quantile(vals, [0.025, 0.975]))
external_subgroups["Macro F1 CI Low"] = np.nan; external_subgroups["Macro F1 CI High"] = np.nan
for idx, r in external_subgroups.iterrows():
    if r["Dataset"] == "Sentiment140":
        part = sbin[sbin.subgroup == r["Subgroup"]]
        lo, hi = boot_ci_binary(part["_y_true"].to_numpy(), part["_y_pred"].to_numpy())
        external_subgroups.loc[idx, ["Macro F1 CI Low", "Macro F1 CI High"]] = [lo, hi]

external_subgroups.to_csv(FINAL["results"] / "F09_F10_external_subgroup_performance.csv", index=False)
display(coverage); display(external_subgroups)

,Dataset,Subgroup,N,Eligible for stable subgroup interpretation
0,SemEval-2014,emoji-heavy,0,False
1,SemEval-2014,slang-heavy,1,False
2,SemEval-2014,sarcasm-indicated,7,False
3,SemEval-2014,reference-unmarked,1532,True
4,Sentiment140,emoji-heavy,72,True
5,Sentiment140,slang-heavy,2795,True
6,Sentiment140,sarcasm-indicated,171,True
7,Sentiment140,reference-unmarked,73512,True


,Dataset,Condition,Subgroup,N,Accuracy,Macro F1,Interpretation Status,Neutral Prediction Rate,Macro F1 Gap vs Reference,Macro F1 CI Low,Macro F1 CI High
0,SemEval-2014,domain/task-shift stress,slang-heavy,1,1.000000,0.333333,exploratory_descriptive_only,NaN,-0.239295,NaN,NaN
1,SemEval-2014,domain/task-shift stress,sarcasm-indicated,7,0.714286,0.488889,exploratory_descriptive_only,NaN,-0.083739,NaN,NaN
2,SemEval-2014,domain/task-shift stress,reference-unmarked,1532,0.646214,0.572628,stable_quantitative,NaN,0.000000,NaN,NaN
3,Sentiment140,binary external robustness,emoji-heavy,72,0.458333,0.369048,stable_quantitative,0.277778,-0.212343,0.283178,0.452197
4,Sentiment140,binary external robustness,slang-heavy,2795,0.538104,0.621341,stable_quantitative,0.271199,0.039951,0.602060,0.638704
5,Sentiment140,binary external robustness,sarcasm-indicated,171,0.555556,0.623291,stable_quantitative,0.228070,0.041901,0.544830,0.701989
6,Sentiment140,binary external robustness,reference-unmarked,73512,0.498898,0.581390,stable_quantitative,0.308997,0.000000,0.577705,0.584600


## 6. Provenance record

In [7]:
prov = {
    "notebook": "FINAL_06_v2",
    "change": "Sentiment140 target replaced: LEGACY s140_test.parquet (10,000 rows) -> frozen EDA_03 v3 sample (100,000 rows)",
    "s140_file_id": S140_FROZEN_FILE_ID, "s140_sha256": S140_SHA256, "s140_rows": int(len(sbin)),
    "s140_prediction_text_column": "text_eval", "s140_tagging_text_column": "text",
    "semeval_file": "LEGACY/data/se_test.parquet", "semeval_unit": "aspect-level row", "semeval_rows": int(len(se3)),
    "locked_model": str(MODEL_PATH), "locked_model_sha256": MODEL_SHA256,
    "sklearn_version": sklearn.__version__,
    "v1_outputs_archived_with_suffix": "_v1_legacy10k",
}
save_json(prov, FINAL["manifests"] / "FINAL_06_v2_PROVENANCE.json")
print(json.dumps(prov, indent=2))

{
  "notebook": "FINAL_06_v2",
  "change": "Sentiment140 target replaced: LEGACY s140_test.parquet (10,000 rows) -> frozen EDA_03 v3 sample (100,000 rows)",
  "s140_file_id": "1Q3FkyjY3tMcAJT6l0Fn9F7MBU5z6MjSk",
  "s140_sha256": "d25327ccb390945e945f687c38f84e3973e81199c7b7977f02d1f5a08593b8ed",
  "s140_rows": 100000,
  "s140_prediction_text_column": "text_eval",
  "s140_tagging_text_column": "text",
  "semeval_file": "LEGACY/data/se_test.parquet",
  "semeval_unit": "aspect-level row",
  "semeval_rows": 1758,
  "locked_model": "/content/drive/MyDrive/LJMU_FINAL_EXPERIMENT_RUN/models/FINAL_SELECTED_PIPELINE.pkl",
  "locked_model_sha256": "bf92c6162dd4664c4a3445d8ce806cc28b09809f75613d6c70f4ef8b6ad954e2",
  "sklearn_version": "1.6.1",
  "v1_outputs_archived_with_suffix": "_v1_legacy10k"
}


## 7. Values to send back for verification
Copy this cell's full output into the chat. Nothing in the thesis is updated until these are checked.

In [8]:
print("=== F09 Sentiment140 ==="); print(pd.read_csv(FINAL["results"]/"F09_Sentiment140_binary_robustness.csv").T.to_string())
print("\n=== F10 SemEval ===");      print(pd.read_csv(FINAL["results"]/"F10_SemEval_domain_task_shift.csv").T.to_string())
print("\n=== Coverage ===");          print(pd.read_csv(FINAL["results"]/"F09_F10_external_subgroup_coverage.csv").to_string(index=False))
print("\n=== Subgroups ===");         print(pd.read_csv(FINAL["results"]/"F09_F10_external_subgroup_performance.csv").to_string(index=False))

=== F09 Sentiment140 ===
                                                                                                           0
Model                                                                                  FinalSelectedPipeline
Target Dataset                                                                                  Sentiment140
Condition                                                                         binary external robustness
Target Sample                                             EDA_03 v3 frozen 100k unique-text sample (50k/50k)
Target N                                                                                              100000
Accuracy                                                                                             0.49134
Macro F1 (negative/positive truth labels)                                                           0.580373
Neutral Prediction Rate                                                                              0.